**Как запустить**

1. Положить рядом с ноутбуком `data/train.csv`, `data/test.csv`, `data/events.csv.gz`, `metric.py`, `sample_submission.csv`.
2. Python 3.13: `pip install -r requirements.txt` (на macOS для LightGBM может понадобиться `brew install libomp`).
3. Запустить Jupyter из папки проекта и выполнить все ячейки по порядку: Kernel → Restart Kernel and Run All Cells. Прогон занимает несколько минут.
4. В конце создаётся `submission.csv`, печатается его MD5 и проверяется, что повторное обучение даёт те же предсказания.

Все модели с фиксированным `random_state`, LightGBM работает в одном потоке с `deterministic=True`. На той же машине результат совпадает полностью; на другой ОС метрика LightGBM может отличаться примерно на 0.01.

Отлавливание парсеров Авито по собятиям cookie

**Задача**  

Нам нужно для каждой cookie_id из test.csv выдать score от 0 до 1: чем выше, тем вероятнее, что это трафик сервиса автоматического сбора данных, и наоборот, чем ниже, тем больше вероятности, что обычный пользователь.

Метрика: максимальные precision среди порогов c recall >= 0.70 (c metric.py)

*Краткое описание подхода и таблица результатов (добавлены сюда в конце)*


In [1]:
import sys

import numpy as np
import pandas as pd

DATA = 'data';

#проверка версий и что, верно все загрузилось
print('python', sys.version.split()[0])
print('pandas', pd.__version__)
print('numpy', np.__version__)

python 3.13.0
pandas 3.0.2
numpy 2.4.4


**1. Загрузка данных**

'day_idx' - номер дня от начала train: 0-13 для train (6-19 апреля), 14-20 для test (20-26 апреля).

In [2]:
DATES = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(f'{DATA}/train.csv', parse_dates=DATES)
test = pd.read_csv(f'{DATA}/test.csv', parse_dates=DATES)
events = pd.read_csv(f'{DATA}/events.csv.gz', parse_dates=['event_ts'])

#номер дня от начала train: 0-13 для train, 14-20 для test
DAY0 = train.window_start_ts.min()
for df in (train, test):
    df['day_idx'] = (df.window_start_ts - DAY0).dt.days

#тут train и test вместе, то есть признаки одинаково для всех cookies
meta = pd.concat([train, test], ignore_index=True)

In [3]:
print('train', train.shape, '| test', test.shape, '| events', events.shape)
print('доля ботов в train:', round(train.target.mean(), 4))
print('train:', train.window_start_ts.min().date(), '-', train.window_start_ts.max().date())
print('test:', test.window_start_ts.min().date(), '-', test.window_start_ts.max().date())
print('длина окна:', (meta.window_end_ts - meta.window_start_ts).unique())
print('повторы cookie_id:', meta.cookie_id.duplicated().sum())
print('кол-во общих cookies train/test:', len(set(train.cookie_id) & set(test.cookie_id)))

train (11091, 6) | test (4909, 5) | events (328905, 14)
доля ботов в train: 0.0811
train: 2026-04-06 - 2026-04-19
test: 2026-04-20 - 2026-04-26
длина окна: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[us]
повторы cookie_id: 0
кол-во общих cookies train/test: 0


In [4]:
#кол-во cookies и какая доля ботов в каждый день train
train.groupby('day_idx').target.agg(n_cookies='size', bot_share='mean').round(3).T

day_idx,0,1,2,3,4,5,6,7,8,9,10,11,12,13
n_cookies,772.000,797.000,834.000,902.000,912.00,896.000,817.000,843.000,826.000,801.000,740.000,713.000,606.000,632.000
bot_share,0.088,0.074,0.084,0.073,0.09,0.088,0.082,0.063,0.085,0.081,0.081,0.093,0.086,0.066


In [5]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


**Выводы**
В train 11091 cookies, доля ботов 8.11%. Классы несбалансированы, constant score дает P@R примерно равно 0.08. Это нижняя планка.  

Train - 6-19 апреля, test - 20-26 апреля, то есть test сторого позже. Валидация должна идти вперед по времени.  
Окно у каждой cookie ровно сутки, одна строка на cookie, пересечений между train и test нет.

**2. Проверка собятий**
Нужна, чтобы исключить учечку данных и ошибки в расчетах.

In [6]:
#берем события cookies из train вместе с их окном и меткой
events_with_dates = events.merge(
    meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
    on='cookie_id',
    how='left',
    validate='many_to_one'
)

missing_cookies = events_with_dates.window_start_ts.isna().sum()
assert missing_cookies == 0, 'Для некотрых событий не найдены границы окна'

#конец окна не включаем: события в этот момент уже недоступны ждя прогноза.
too_early = events_with_dates.event_ts < events_with_dates.window_start_ts
too_late  = events_with_dates.event_ts >= events_with_dates.window_end_ts
inside_window = ~too_early & ~too_late

print('До начала окна:', too_early.sum())
print('Внутри окна:', inside_window.sum())
print('После окончания окна:', too_late.sum())
print(f'Доля поздних событий: {too_late.mean():.1%}')

До начала окна: 0
Внутри окна: 288126
После окончания окна: 40779
Доля поздних событий: 12.4%


In [7]:
train_events = events_with_dates.merge(
    train[['cookie_id', 'target']],
    on='cookie_id',
    how='inner',
    validate='many_to_one'
)

train_events['late_event'] = (
    train_events.event_ts >= train_events.window_end_ts
)

print(f'Поздних событий в train: {train_events.late_event.mean():.1%}')
late_by_class = train_events.groupby('target').late_event.agg(
    total='size',
    late='sum',
    share='mean'
)
print('Поздние события по классам:')
print(late_by_class.round(3))

#Отдельно проверяем капчу: она может выдать информацию о боте из будущего.
captcha_events = train_events[
    train_events.event_name == 'captcha_shown'
]
print('Показы капчи:')
print(captcha_events.groupby(['target', 'late_event']).size())

Поздних событий в train: 17.0%
Поздние события по классам:
         total   late  share
target                      
0       193135  21206  0.110
1        46080  19573  0.425
Показы капчи:
target  late_event
0       True           958
1       True          6970
dtype: int64


In [8]:
#проверяем дубли и порядок событий
duplicate_rows = events.duplicated()

print('Полных дублей:', duplicate_rows.sum())
print(f'Доля дублей: {duplicate_rows.mean():.2%}')

train_events['duplicate_row'] = train_events.duplicated(
    subset=events.columns
)
print('Доля дублей по классам, %:')
print(
    (train_events.groupby('target').duplicate_row.mean() * 100).round(2)
)
#diff пока применяется к исходному порядку, чтобы найти нарушения.
time_difference = events.groupby('cookie_id', sort=False).event_ts.diff()
wrong_order = time_difference < pd.Timedelta(0)

print('Файл отсортирован по времени:', events.event_ts.is_monotonic_increasing)
print('Нарушений порядка внутри cookie:', wrong_order.sum())

Полных дублей: 4863
Доля дублей: 1.48%
Доля дублей по классам, %:
target
0    1.48
1    1.42
Name: duplicate_row, dtype: float64
Файл отсортирован по времени: False
Нарушений порядка внутри cookie: 156231


In [9]:
print('Исходных вариантов:', events.platform.nunique())
print(events.platform.value_counts().sort_index())

platform_fixed = (
    events.platform.str.strip().str.lower().replace({'desktop': 'web', 'iphone': 'ios'})
)

print('После объяединения:')
print(platform_fixed.value_counts())

unknown_platforms = ~platform_fixed.isin(['web', 'android', 'ios'])
print('Неизвестных платформ:', unknown_platforms.sum())

Исходных вариантов: 11
platform
ANDROID    42462
Android    42254
IOS         4100
WEB        46476
Web        46365
android    42159
desktop    46866
iOS         4091
ios         4078
iphone      4103
web        45951
Name: count, dtype: int64
После объяединения:
platform
web        185658
android    126875
ios         16372
Name: count, dtype: int64
Неизвестных платформ: 0


In [10]:
#проверяем пропуски
missing_percent = events.isna().mean() * 100

print('Пропуски по столбцам, %:')
print(missing_percent[missing_percent > 0].sort_values(ascending=False).round(1))

web_events = events[platform_fixed == 'web']
mobile_events = events[platform_fixed != 'web']
search_events = events[events.event_name == 'search_results_view']

print(f'Без координаты курсора на web: {web_events.pointer_x.isna().mean():.1%}')
print(f'Без координаты курсора на mobile: {mobile_events.pointer_x.isna().mean():.1%}')
print(f'Страница заполнена у поисковых событий: {search_events.search_page.notna().mean():.1%}')

Пропуски по столбцам, %:
search_query     69.5
search_page      69.5
pointer_x        67.0
pointer_y        67.0
seller_type      40.7
item_id          34.8
item_category    10.0
item_location     7.2
dtype: float64
Без координаты курсора на web: 41.5%
Без координаты курсора на mobile: 100.0%
Страница заполнена у поисковых событий: 100.0%


**Выводы**  

12.4% всех событий находятся после оконачания окна. Среди событий train таких 17%, причем у ботов доля выше. Надо удалить все эти события, чтобы избежать утечки данных.   
События не отсортированы по времени. 
Прлпуски во многом зависят от типа события и платформы, нельзя просто заполнить всё.

**3. Очистка событий**  

Удаляю дубли и события вне окна наблюдения, привожу названия платформ к одному виду и сортирую события по времени. Очищенные данные дальше использую для расчет поведенческих признаков.

In [11]:
#удаляем дубли и события вне окна
events_clean = events_with_dates.drop_duplicates(
    subset=events.columns).copy()

print('После удаления дублей:', len(events_clean))
#Для прогнозы достпуны только события, произошедшие внутри окна конкретной cookie.
events_clean = events_clean[(events_clean.event_ts >= events_clean.window_start_ts) & (events_clean.event_ts < events_clean.window_end_ts)].copy()

print('После ограничения по времени:', len(events_clean))
print('Удалено всего:', len(events) - len(events_clean))

После удаления дублей: 324042
После ограничения по времени: 283833
Удалено всего: 45072


In [12]:
#приводим platform к одному виду
events_clean['platform'] = (
    events_clean.platform.str.strip().str.lower().replace({'desktop': 'web', 'iphone': 'ios'})
)
events_clean = events_clean.merge(
    meta[['cookie_id', 'day_idx']],
    on='cookie_id',
    how='left',
    validate='many_to_one'
)
print('Платформы:')
print(events_clean.platform.value_counts())
print('Неизвестных платформ:', (~events_clean.platform.isin(['web', 'android', 'ios'])).sum())
print('Событий без day_idx:', events_clean.day_idx.isna().sum())
    

Платформы:
platform
web        158898
android    110556
ios         14379
Name: count, dtype: int64
Неизвестных платформ: 0
Событий без day_idx: 0


In [13]:
#через total_seconds(), чтобы результат не зависел от разрешения datetime в pandas
events_clean['t'] = (
    events_clean.event_ts - pd.Timestamp('1970-01-01')).dt.total_seconds().astype('int64')
#при одинаковом времени используем eid для устойчивого порядка событий
events_clean = events_clean.sort_values(
    ['cookie_id', 't', 'eid'],
    kind='mergesort').reset_index(drop=True)

print(events_clean[['cookie_id', 'event_ts', 't', 'eid']].head())

             cookie_id            event_ts           t  eid
0  ck_00013fdfa0bd37dd 2026-04-24 18:41:16  1777056076  210
1  ck_00013fdfa0bd37dd 2026-04-24 19:16:53  1777058213  200
2  ck_00013fdfa0bd37dd 2026-04-24 19:16:54  1777058214  210
3  ck_00013fdfa0bd37dd 2026-04-24 19:17:05  1777058225  200
4  ck_00013fdfa0bd37dd 2026-04-24 20:05:05  1777061105  500


In [14]:
#проверяем результат очистки
within_window = (
    (events_clean.event_ts >= events_clean.window_start_ts)
    & (events_clean.event_ts < events_clean.window_end_ts)
)

time_gaps = events_clean.groupby('cookie_id').t.diff()

assert within_window.all(), 'Остались события за пределами окна'
assert (time_gaps.dropna() >= 0).all(), 'Нарушен порядок событий'
assert events_clean.platform.isin(['web', 'android', 'ios']).all()
assert events_clean.day_idx.notna().all()
assert events_clean.cookie_id.nunique() == meta.cookie_id.nunique()

print('Все проверки пройдены')
print('Событий после очистки:', len(events_clean))
print('Уникальных cookie:', events_clean.cookie_id.nunique())
print('Пропусков в ключевых полях:')
print(
    events_clean[['cookie_id', 'event_ts', 't', 'eid', 'day_idx']].isna().sum())

Все проверки пройдены
Событий после очистки: 283833
Уникальных cookie: 16000
Пропусков в ключевых полях:
cookie_id    0
event_ts     0
t            0
eid          0
day_idx      0
dtype: int64


**4. Признаки по типам событий**  

Считаю для каждой cookie количество и долю разных действий, а также несколько отношений к числу просмотров объявленний.

In [15]:
event_counts = events_clean.event_name.value_counts()

print(event_counts)
print('Типов событий:', events_clean.event_name.nunique())

event_name
item_view               106448
search_results_view      88479
photo_swipe              32570
favorite_add             17012
seller_page_view         15343
contact_phone_show       10145
login                     5579
contact_chat_open         5479
contact_message_sent      2778
Name: count, dtype: int64
Типов событий: 9


In [16]:
event_types = [
    'search_results_view',
    'item_view',
    'photo_swipe',
    'seller_page_view',
    'contact_phone_show',
    'contact_chat_open',
    'contact_message_sent',
    'favorite_add',
    'login'
]
missing_event_types = set(event_types) - set(events_clean.event_name.unique())
print('Не найдено типов событий:', missing_event_types)

Не найдено типов событий: set()


In [17]:
#считаем кол_во каждого действия для каждой cookie
events_per_cookie = events_clean.groupby('cookie_id').size()

event_table = pd.crosstab(
    events_clean.cookie_id,
    events_clean.event_name
).reindex(columns=event_types, fill_value=0)

activity_features = pd.DataFrame(index=events_per_cookie.index)
activity_features['n_events'] = events_per_cookie

for event_name in event_types:
    activity_features[f'cnt_{event_name}'] = event_table[event_name]

print('Размер таблицы:', activity_features.shape)
activity_features.head()

Размер таблицы: (16000, 10)


,n_events,cnt_search_results_view,cnt_item_view,cnt_photo_swipe,cnt_seller_page_view,cnt_contact_phone_show,cnt_contact_chat_open,cnt_contact_message_sent,cnt_favorite_add,cnt_login
cookie_id,,,,,,,,,,
ck_00013fdfa0bd37dd,15,3,4,3,0,0,2,0,1,2
ck_000c95f1408dcb00,16,3,11,1,1,0,0,0,0,0
ck_000e8c52636e3bec,34,9,11,5,3,1,0,0,4,1
ck_0010e31baa4a1fb7,16,4,7,1,1,0,0,0,2,1
ck_0010ec3874fb5378,74,20,25,14,4,0,0,0,10,1


In [18]:
#считаем долю каждого типа события
for event_name in event_types:
    activity_features[f'share_{event_name}'] = (
        activity_features[f'cnt_{event_name}']
        / activity_features['n_events']
    )
share_columns = [
    column for column in activity_features.columns
    if column.startswith('share_')
]
print('Добавлено долей:', len(share_columns))
print('Минимальная сумма долей:', activity_features[share_columns].sum(axis=1).min())
print(
    'Максимальная сумма долей:',
    activity_features[share_columns].sum(axis=1).max()
)

Добавлено долей: 9
Минимальная сумма долей: 0.9999999999999997
Максимальная сумма долей: 1.0000000000000002


In [19]:
#изучаю отношения действия к просмотрам
item_views = activity_features['cnt_item_view'].replace(0, np.nan)
search_views = activity_features['cnt_search_results_view'].replace(0, np.nan)

activity_features['photo_per_view'] = (
    activity_features['cnt_photo_swipe'] / item_views
)

activity_features['fav_per_view'] = (
    activity_features['cnt_favorite_add'] / item_views
)

activity_features['seller_per_view'] = (
    activity_features['cnt_seller_page_view'] / item_views
)

activity_features['contact_per_view'] = (
    activity_features['cnt_contact_phone_show']
    + activity_features['cnt_contact_chat_open']
) / item_views

activity_features['view_per_search'] = (
    activity_features['cnt_item_view'] / search_views
)

print('Размер таблицы:', activity_features.shape)

Размер таблицы: (16000, 24)


In [20]:
print('Cookie в признаках:', len(activity_features))
print('Столбцов:', activity_features.shape[1])

missing_values = activity_features.isna().sum()
print('Пропуски:')
print(missing_values[missing_values > 0])

activity_features[
    [
        'n_events',
        'cnt_item_view',
        'share_item_view',
        'photo_per_view',
        'contact_per_view',
        'view_per_search'
    ]
].describe().round(3)

Cookie в признаках: 16000
Столбцов: 24
Пропуски:
photo_per_view      1036
fav_per_view        1036
seller_per_view     1036
contact_per_view    1036
view_per_search     1356
dtype: int64


,n_events,cnt_item_view,share_item_view,photo_per_view,contact_per_view,view_per_search
count,16000.000,16000.000,16000.000,14964.000,14964.000,14644.000
mean,17.740,6.653,0.370,0.395,0.160,1.398
std,17.729,7.288,0.183,0.499,0.288,1.169
min,1.000,0.000,0.000,0.000,0.000,0.000
25%,6.000,2.000,0.273,0.000,0.000,0.750
50%,12.000,4.000,0.364,0.273,0.000,1.078
75%,23.000,9.000,0.471,0.500,0.222,1.733
max,187.000,96.000,1.000,8.000,4.500,15.000


In [21]:
assert activity_features.index.is_unique
assert len(activity_features) == meta.cookie_id.nunique()
assert activity_features['n_events'].sum() == len(events_clean)
print('Проверки пройдены')
print('Всего событий учтено:', activity_features.n_events.sum())

Проверки пройдены
Всего событий учтено: 283833


**Вывод**

Получила 24 признака по составу событиий: кол-ва и доли действий, а также отношения признаков к просмотрам.

**5. Интервалы и сессии**  

Рассчитываю интервалы между действиями внутри сессий, их разброс и повторяемость. Также учитываю число сессий, длительность активности и ночные события. Новая сессия начинается после паузы больше 30 минут.

In [22]:
SESSION_GAP = 30 * 60

steps = events_clean[['cookie_id', 't', 'event_ts']].copy()
steps['dt'] = steps.groupby('cookie_id').t.diff()

# паузы больше 30 минут не учитываю при расчёте темпа внутри сессии
steps['new_session'] = steps.dt.isna() | (steps.dt > SESSION_GAP)
steps['session'] = steps.groupby('cookie_id').new_session.cumsum()
steps['dt_in'] = steps.dt.where(~steps.new_session)

# сравниваюю соседние интервалы, не переходя через границу сессии
steps['gap_change'] = steps.groupby('cookie_id').dt_in.diff().abs()

print('Всего сессий:', steps.new_session.sum())
print('Интервалов внутри сессий:', steps.dt_in.notna().sum())
print('Пар соседних интервалов:', steps.gap_change.notna().sum())

Всего сессий: 47288
Интервалов внутри сессий: 236545
Пар соседних интервалов: 197537


In [23]:
#распределяю интервалы
inside = steps.dropna(subset=['dt_in'])
gaps = inside.groupby('cookie_id')['dt_in']

time_features = pd.DataFrame({'dt_min': gaps.min()})

for q in (0.1, 0.25, 0.5, 0.75, 0.9):
    time_features[f'dt_q{int(q * 100)}'] = gaps.quantile(q)

time_features['dt_mean'] = gaps.mean()
time_features['dt_cv'] = gaps.std() / gaps.mean()
time_features['logdt_std'] = np.log1p(inside.dt_in).groupby(inside.cookie_id).std()
time_features['dt_unique_ratio'] = gaps.nunique() / gaps.size()

# доля коротких и длинных пауз внутри сессии
for start, end in [(0, 2), (2, 5), (5, 12), (12, 30), (30, 90), (90, SESSION_GAP)]:
    time_features[f'dtbin_{start}_{end}'] = (
        inside.dt_in.between(start, end, inclusive='right')
        .groupby(inside.cookie_id).mean()
    )

print('Размер таблицы:', time_features.shape)
print('Медиана интервалов по всем cookie:', time_features.dt_q50.median())
print(time_features[['dt_min', 'dt_q50', 'dt_mean']].head(3).round(2))

Размер таблицы: (15513, 16)
Медиана интервалов по всем cookie: 46.0
                     dt_min  dt_q50  dt_mean
cookie_id                                   
ck_00013fdfa0bd37dd     1.0   111.0   219.42
ck_000c95f1408dcb00    17.0    34.0    39.00
ck_000e8c52636e3bec     0.0    21.5    75.75


In [24]:
neighbours = steps.dropna(subset=['gap_change'])

time_features['near_const_dt_share'] = (
    (neighbours.gap_change <= 1)
    .groupby(neighbours.cookie_id).mean()
)

# короткие сессии исключаю: по паре интервалов сложно судить о регулярности
runs = inside.groupby(['cookie_id', 'session']).dt_in.agg(
    ['median', 'mean', 'std', 'size']
)

runs = runs[runs['size'] >= 4]

time_features['sess_min_median_dt'] = (runs['median'].groupby(level=0).min())
time_features['sess_min_cv'] = (
    (runs['std'] / runs['mean']).groupby(level=0).min()
)

print('Сессий с 4+ интервалами:', len(runs))
print('Размер таблицы:', time_features.shape)
print(
    time_features[
        ['dt_q50', 'near_const_dt_share', 'sess_min_median_dt']
    ].head(3).round(3)
)

Сессий с 4+ интервалами: 21757
Размер таблицы: (15513, 19)
                     dt_q50  near_const_dt_share  sess_min_median_dt
cookie_id                                                           
ck_00013fdfa0bd37dd   111.0                0.000               148.5
ck_000c95f1408dcb00    34.0                0.167                46.5
ck_000e8c52636e3bec    21.5                0.033                23.0


In [25]:
#определяю длительность, активность и ночные события

sessions = steps.groupby(['cookie_id', 'session']).t.agg(['min', 'max', 'size'])
sessions['dur'] = sessions['max'] - sessions['min']
per_cookie = sessions.groupby(level=0)

time_features['n_sessions'] = per_cookie.size()
time_features['sess_len_max'] = per_cookie['size'].max()
time_features['sess_len_mean'] = per_cookie['size'].mean()
time_features['sess_dur_max'] = per_cookie.dur.max()
time_features['active_time'] = per_cookie.dur.sum()

# +1 минута защищает от деления на ноль при мгновенных сессиях
time_features['events_per_min'] = (
    per_cookie['size'].sum()
    / (time_features.active_time / 60 + 1)
)

cookie_times = steps.groupby('cookie_id').t
time_features['span'] = cookie_times.max() - cookie_times.min()

hours = steps.event_ts.dt.hour
time_features['night_share'] = (hours < 6).groupby(steps.cookie_id).mean()
time_features['n_hours'] = hours.groupby(steps.cookie_id).nunique()
time_features['first_hour'] = hours.groupby(steps.cookie_id).first()

print('Размер таблицы:', time_features.shape)
print(
    time_features[
        ['dt_q50', 'n_sessions', 'near_const_dt_share', 'events_per_min']
    ].describe().round(3)
)

Размер таблицы: (15513, 29)
          dt_q50  n_sessions  near_const_dt_share  events_per_min
count  15513.000   15513.000            14550.000       15513.000
mean      67.065       3.007                0.033           1.232
std       92.124       2.136                0.085           1.058
min        0.000       1.000                0.000           0.060
25%       29.000       1.000                0.000           0.615
50%       46.000       2.000                0.000           0.996
75%       75.000       4.000                0.033           1.543
max     1759.000      17.000                1.000          18.412


In [26]:
assert time_features.index.is_unique
assert time_features.columns.is_unique
assert time_features.index.isin(meta.cookie_id).all()
assert (steps.dt.dropna() >= 0).all()
assert time_features.shape[1] == 29
assert not np.isinf(time_features.to_numpy(dtype=float)).any()

print('Все проверки пройдены')
print('Кук с временными признаками:', len(time_features))
print('Кук без интервалов внутри сессий:', len(meta) - len(time_features))
print('Признаков:', time_features.shape[1])
print('Пропусков в near_const_dt_share:',
      time_features.near_const_dt_share.isna().sum())
print('Пропусков в sess_min_median_dt:',
      time_features.sess_min_median_dt.isna().sum())

Все проверки пройдены
Кук с временными признаками: 15513
Кук без интервалов внутри сессий: 487
Признаков: 29
Пропусков в near_const_dt_share: 963
Пропусков в sess_min_median_dt: 3545


**Вывод**  

Получила 29 временных признаков для 15513 cookie. У 487 cookie нет интервалов внутри сессий, их верну при общей сборке табице признаков. Длительные паузы между сессиями не использую при расчета обычного темпа действий.

**6. Поиск, объявления, курсор и клиент**  

Дополняю поведенческие признаки глубиной поиска, разнобразием объявлений и городов, движением курсора, платформой и возрастом cookie. Все признаки рассчитываю по событиям внутри окна наблюдения.

In [27]:
#изучаю поведение в поиске
search = events_clean.loc[
    events_clean.eid == 100,
    ['cookie_id', 'search_query', 'search_page']
].copy()

by_cookie = search.groupby('cookie_id')

search_feat = pd.DataFrame({
    'page_max': by_cookie.search_page.max(),
    'page_mean': by_cookie.search_page.mean()
})

search_feat['page_share_gt5'] = (
    (search.search_page > 5).groupby(search.cookie_id).mean()
)
search_feat['n_queries'] = by_cookie.search_query.nunique()
search_feat['search_per_query'] = (
    by_cookie.size() / search_feat.n_queries
)

prev_q = by_cookie.search_query.shift()
prev_page = by_cookie.search_page.shift()

# последовательное листание считаю только внутри одного запроса
same_q = search.search_query == prev_q
next_page = search.search_page == prev_page + 1

search_feat['page_step_share'] = (
    (same_q & next_page).groupby(search.cookie_id).mean()
)

# для скачков сохраняю все соседние поисковые события, даже при смене запроса
search_feat['page_jump_mean'] = (
    (search.search_page - prev_page).abs()
    .groupby(search.cookie_id).mean()
)

print('Поисковых событий:', len(search))
print('Размер таблицы:', search_feat.shape)
print(
    search_feat[
        ['page_max', 'n_queries', 'page_step_share', 'page_jump_mean']
    ].head(3).round(3)
)

Поисковых событий: 88479
Размер таблицы: (14644, 7)
                     page_max  n_queries  page_step_share  page_jump_mean
cookie_id                                                                
ck_00013fdfa0bd37dd       3.0          3              0.0           1.000
ck_000c95f1408dcb00       4.0          3              0.0           2.000
ck_000e8c52636e3bec      12.0          8              0.0           3.875


In [28]:
#теперь анализирую объявления

items = events_clean[events_clean.item_id.notna()]
item_groups = items.groupby('cookie_id')

item_feat = pd.DataFrame({
    'n_items': item_groups.item_id.nunique()
})

item_feat['item_repeat'] = (
    1 - item_feat.n_items / item_groups.size()
)

item_feat['pro_share'] = (
    (items.seller_type == 'pro')
    .groupby(items.cookie_id).mean()
)

print('Событий с item_id:', len(items))
print('Размер таблицы:', item_feat.shape)
print(item_feat.head(3).round(3))

Событий с item_id: 189775
Размер таблицы: (15762, 3)
                     n_items  item_repeat  pro_share
cookie_id                                           
ck_00013fdfa0bd37dd        9        0.100      0.200
ck_000c95f1408dcb00        9        0.308      0.077
ck_000e8c52636e3bec       20        0.167      0.333


In [29]:
#работаю с категориями и географией

all_cookie = events_clean.groupby('cookie_id')

item_feat['n_cats'] = all_cookie.item_category.nunique()
item_feat['n_locs'] = all_cookie.item_location.nunique()
item_feat['locs_per_event'] = (
    item_feat.n_locs / all_cookie.size()
)

# доля самого частого города и категории среди заполненных значений
for col, name in [
    ('item_location', 'loc_top_share'),
    ('item_category', 'cat_top_share')
]:
    counts = (
        events_clean.dropna(subset=[col])
        .groupby(['cookie_id', col]).size()
    )
    item_feat[name] = (
        counts.groupby(level=0).max()
        / counts.groupby(level=0).sum()
    )

locations = events_clean[events_clean.item_location.notna()]

prev_loc = locations.groupby('cookie_id').item_location.shift()
changed = locations.item_location != prev_loc

# первое наблюдение тоже считаю как переход, как в исходном решении
item_feat['loc_switch_share'] = (
    changed.groupby(locations.cookie_id).mean()
)

print('Размер таблицы:', item_feat.shape)
print(
    item_feat[
        ['n_items', 'n_locs', 'loc_top_share', 'loc_switch_share']
    ].head(3).round(3)
)

Размер таблицы: (15762, 9)
                     n_items  n_locs  loc_top_share  loc_switch_share
cookie_id                                                            
ck_00013fdfa0bd37dd        9       4          0.538             0.692
ck_000c95f1408dcb00        9       8          0.286             0.786
ck_000e8c52636e3bec       20      12          0.281             1.000


In [30]:
#изучаю поведения курсора

has_pos = events_clean.pointer_x.notna()

pointer_feat = pd.DataFrame({
    'ptr_share': has_pos.groupby(events_clean.cookie_id).mean()
})

web = events_clean.platform == 'web'

pointer_feat['ptr_missing_web'] = (
    (~has_pos[web])
    .groupby(events_clean.cookie_id[web]).mean()
)

cursor = events_clean[has_pos]
by_cookie = cursor.groupby('cookie_id')

pointer_feat['ptr_x_std'] = by_cookie.pointer_x.std()
pointer_feat['ptr_y_std'] = by_cookie.pointer_y.std()
pointer_feat['ptr_x_mean'] = by_cookie.pointer_x.mean()
pointer_feat['ptr_x_max'] = by_cookie.pointer_x.max()

pointer_feat['ptr_uniq'] = (
    by_cookie.pointer_x.nunique() / by_cookie.size()
)

# расстояние между соседними позициями курсора одной cookie
dx = cursor.pointer_x - by_cookie.pointer_x.shift()
dy = cursor.pointer_y - by_cookie.pointer_y.shift()
step = np.hypot(dx, dy)

pointer_feat['ptr_step_med'] = (
    step.groupby(cursor.cookie_id).median()
)

pointer_feat['ptr_center_dist'] = (
    np.hypot(cursor.pointer_x - 960, cursor.pointer_y - 540)
    .groupby(cursor.cookie_id).mean()
)

pointer_feat['ptr_grid_share'] = (
    ((cursor.pointer_x % 10 == 0) & (cursor.pointer_y % 10 == 0))
    .groupby(cursor.cookie_id).mean()
)

print('Событий с курсором:', len(cursor))
print('Размер таблицы:', pointer_feat.shape)
print(
    pointer_feat[
        ['ptr_share', 'ptr_missing_web', 'ptr_x_std', 'ptr_step_med']
    ].head(3).round(3)
)

Событий с курсором: 94258
Размер таблицы: (16000, 10)
                     ptr_share  ptr_missing_web  ptr_x_std  ptr_step_med
cookie_id                                                               
ck_00013fdfa0bd37dd      0.000              NaN        NaN           NaN
ck_000c95f1408dcb00      1.000            0.000    494.981       556.059
ck_000e8c52636e3bec      0.912            0.088    561.544       794.766


In [31]:
#расчитываю возраст cookie, доли разных платформ и число различных user-agent

cookie_info = meta.set_index('cookie_id')
client_feat = pd.DataFrame(index=cookie_info.index)

age_hours = (
    cookie_info.window_start_ts - cookie_info.cookie_created_at
).dt.total_seconds() / 3600

client_feat['cookie_age_log'] = np.log1p(age_hours)

platform_share = pd.crosstab(
    events_clean.cookie_id,
    events_clean.platform,
    normalize='index'
)

for platform in ('web', 'android', 'ios'):
    client_feat[f'platform_{platform}'] = (
        platform_share.get(platform, 0.0)
    )

by_cookie = events_clean.groupby('cookie_id')

client_feat['n_platforms'] = by_cookie.platform.nunique()
client_feat['n_user_agents'] = by_cookie.user_agent.nunique()

print('Размер таблицы:', client_feat.shape)
print(client_feat.sort_index().head(3).round(3))

Размер таблицы: (16000, 6)
                     cookie_age_log  platform_web  platform_android  \
cookie_id                                                             
ck_00013fdfa0bd37dd           8.457           0.0               1.0   
ck_000c95f1408dcb00           8.629           1.0               0.0   
ck_000e8c52636e3bec           6.418           1.0               0.0   

                     platform_ios  n_platforms  n_user_agents  
cookie_id                                                      
ck_00013fdfa0bd37dd           0.0            1              1  
ck_000c95f1408dcb00           0.0            1              1  
ck_000e8c52636e3bec           0.0            1              1  


In [32]:
#общая проверка этапа

groups = {
    'search': search_feat,
    'items': item_feat,
    'pointer': pointer_feat,
    'client': client_feat
}

all_cols = list(activity_features.columns) + list(time_features.columns)

for name, data in groups.items():
    assert data.index.is_unique, f'Дубли cookie в {name}'
    assert data.index.isin(meta.cookie_id).all()
    all_cols.extend(data.columns)
    print(
        f'{name}: {len(data)} cookie, '
        f'{data.shape[1]} признаков, '
        f'без данных: {len(meta) - len(data)}'
    )

assert len(all_cols) == len(set(all_cols)), 'Повторяются названия признаков'

print('Новых признаков:', sum(data.shape[1] for data in groups.values()))
print('Всего признаков на данный момент:', len(all_cols))
print('Все проверки пройдены')

search: 14644 cookie, 7 признаков, без данных: 1356
items: 15762 cookie, 9 признаков, без данных: 238
pointer: 16000 cookie, 10 признаков, без данных: 0
client: 16000 cookie, 6 признаков, без данных: 0
Новых признаков: 32
Всего признаков на данный момент: 85
Все проверки пройдены


**Вывод**  


Добавила 32 признака: поиск (7), объявления и география (9), курсор (10), клиент (6). Всего получила 85 поведенческих признаков. Отсутствующие характеристики оставляю как NaN, чтобы не подменять их искусственными значениями.

**7. Общие объявления в пределах одного дня**

Проверяю, пересекаются ли cookie по просмотренным объявлениям. Считаю число общих объявлений, количество других cookie с похожей активностью и популярность просмотренных объявлений за тот же день.

In [33]:
from scipy import sparse

pairs = events_clean.loc[
    events_clean.item_id.notna(),
    ['cookie_id', 'item_id', 'day_idx']
].drop_duplicates()

print('Уникальных пар:', len(pairs))
print('Cookie с объявлениями:', pairs.cookie_id.nunique())
print('Разных объявлений:', pairs.item_id.nunique())
print('Дней:', pairs.day_idx.nunique())

Уникальных пар: 149855
Cookie с объявлениями: 15762
Разных объявлений: 52929
Дней: 21


In [34]:
#считаю пересечения между cookies
day_parts = []

for day, today in pairs.groupby('day_idx'):
    cookie_no, cookies = pd.factorize(today.cookie_id)
    ad_no, _ = pd.factorize(today.item_id)
    
    # разреженная матрица: не храним все нулевые значения
    visits = sparse.csr_matrix(
        (np.ones(len(today)), (cookie_no, ad_no))
    )
    
    visitors = np.asarray(visits.sum(axis=0)).ravel()
    shared = (visits @ visits.T).tolil()
    
    # саму cookie не считаем её соседом
    shared.setdiag(0)
    shared = shared.tocsr()
    today = today.assign(visitors=visitors[ad_no])
    day_feat = pd.DataFrame({
        'max_shared_items': np.asarray(
            shared.max(axis=1).todense()
        ).ravel(),
        'n_neighbors_1': np.asarray(
            (shared >= 1).sum(axis=1)
        ).ravel(),
        'n_neighbors_2': np.asarray(
            (shared >= 2).sum(axis=1)
        ).ravel()
    }, index=cookies)
    day_feat['item_pop_log_mean'] = (
        np.log(today.visitors)
        .groupby(today.cookie_id).mean()
    )
    day_feat['item_solo_share'] = (
        (today.visitors == 1)
        .groupby(today.cookie_id).mean()
    )
    day_parts.append(day_feat)
    
covisit_feat = pd.concat(day_parts)

print('Обработано дней:', len(day_parts))
print('Размер таблицы:', covisit_feat.shape)
print(covisit_feat.sort_index().head(3).round(3))

Обработано дней: 21
Размер таблицы: (15762, 5)
                     max_shared_items  n_neighbors_1  n_neighbors_2  \
ck_00013fdfa0bd37dd               1.0              1              0   
ck_000c95f1408dcb00               1.0              1              0   
ck_000e8c52636e3bec               1.0              1              0   

                     item_pop_log_mean  item_solo_share  
ck_00013fdfa0bd37dd              0.077            0.889  
ck_000c95f1408dcb00              0.077            0.889  
ck_000e8c52636e3bec              0.035            0.950  


In [35]:
assert covisit_feat.index.is_unique
assert covisit_feat.index.isin(meta.cookie_id).all()
assert (covisit_feat.n_neighbors_2 <= covisit_feat.n_neighbors_1).all()
assert covisit_feat.item_solo_share.between(0, 1).all()

# если есть сосед с двумя общими объявлениями, максимум не может быть меньше двух
assert (
    (covisit_feat.max_shared_items >= 2)
    == (covisit_feat.n_neighbors_2 > 0)
).all()

print('Cookie без общих объявлений:',
      (covisit_feat.n_neighbors_1 == 0).sum())

print('Cookie с двумя и более общими объявлениями у соседа:',
      (covisit_feat.n_neighbors_2 > 0).sum())

print('Максимум общих объявлений:',
      covisit_feat.max_shared_items.max())

print('Cookie без item_id:', len(meta) - len(covisit_feat))
print('Все проверки пройдены')

Cookie без общих объявлений: 6650
Cookie с двумя и более общими объявлениями у соседа: 570
Максимум общих объявлений: 8.0
Cookie без item_id: 238
Все проверки пройдены


In [36]:
#собираю все поведенческие признаки

feat_groups = {
    'activity': activity_features,
    'timing': time_features,
    'search': search_feat,
    'items': item_feat,
    'pointer': pointer_feat,
    'client': client_feat,
    'covisit': covisit_feat
}

all_cols = []
for name, data in feat_groups.items():
    assert data.index.is_unique, f'Повторяются cookie в {name}'
    all_cols.extend(data.columns)
    print(name, data.shape[1])
assert len(all_cols) == len(set(all_cols))

# выравниваем все группы по одному списку cookie
base_features = pd.concat(
    [data.reindex(meta.cookie_id) for data in feat_groups.values()],
    axis=1
)
base_features = base_features.replace([np.inf, -np.inf], np.nan)

assert base_features.shape == (16000, 90)
assert base_features.index.is_unique
assert 'target' not in base_features.columns

print('Общая таблица:', base_features.shape)
print('Train:', base_features.loc[train.cookie_id].shape)
print('Test:', base_features.loc[test.cookie_id].shape)
print('Все проверки пройдены')

activity 24
timing 29
search 7
items 9
pointer 10
client 6
covisit 5
Общая таблица: (16000, 90)
Train: (11091, 90)
Test: (4909, 90)
Все проверки пройдены


**Вывод**

Добавила 5 признаков по общим объявлениям внутри одного дня. Получила общую таблицу из 90 поведенческих признаков для 16 000 cookie.

Для расчёта пересечений использовала только события текущего суточного окна, без target. При сборке сохранила cookie без отдельных видов активности и оставила недостающие значения как NaN.

**8. Репутация объявлений**  

Считаю, как часто просмотренные cookie объявления встречались у известных ботов в прошлые дни. Для каждого объявления использую только историю, доступную до текущего дня. В валидации дополнительно ограничиваю историю обучающим периодом.

In [37]:
#готовлю историю просмотра
seen = events_clean.loc[
    events_clean.item_id.notna(),
    ['cookie_id', 'item_id']
].drop_duplicates()

first_week = train[train.day_idx < 7].copy()

print('Уникальных пар:', len(seen))
print('Cookie с объявлениями:', seen.cookie_id.nunique())
print('Размеченных cookie за первую неделю:', len(first_week))
print('Доля ботов:', round(first_week.target.mean(), 4))

Уникальных пар: 149855
Cookie с объявлениями: 15762
Размеченных cookie за первую неделю: 5930
Доля ботов: 0.0828


In [38]:
#тут я собираю статистику объявлений по дням
def item_stats(known):
    visits = seen.merge(
        known[['cookie_id', 'day_idx', 'target']],
        on='cookie_id',
        how='inner',
        validate='many_to_one'
    )

    return (
        visits.groupby(['item_id', 'day_idx'])
        .target.agg(bots='sum', visitors='size')
        .reset_index()
    )

week_stats = item_stats(first_week)

print('Строк в истории:', len(week_stats))
print('Разных объявлений:', week_stats.item_id.nunique())
print('Дней:', week_stats.day_idx.nunique())
print('Посещений ботами:', week_stats.bots.sum())

Строк в истории: 50750
Разных объявлений: 34332
Дней: 7
Посещений ботами: 6915


In [39]:
#считаю признаки репутации
def item_history(who, known, stop_day, alpha=2.0):
    daily = item_stats(known)

    wanted = seen.merge(
        who[['cookie_id', 'day_idx']],
        on='cookie_id',
        validate='many_to_one'
    )
    wanted['limit'] = np.minimum(wanted.day_idx, stop_day)
    hits = wanted[['cookie_id', 'item_id', 'limit']].merge(
        daily, on='item_id', how='left'
    )

    # беру историю объявления строго до текущего дня и до начала валидации
    hits = hits[
        hits.day_idx.isna() | (hits.day_idx < hits.limit)
    ]
    totals = (
        hits.groupby(['cookie_id', 'item_id'])[['bots', 'visitors']]
        .sum()
        .reset_index()
    )
    
    totals = wanted[['cookie_id', 'item_id']].merge(
        totals, on=['cookie_id', 'item_id'], how='left'
    ).fillna({'bots': 0, 'visitors': 0})

    # сглаживаю долю ботов: одного посещения мало для надёжной оценки
    p0 = known.target.mean()
    totals['rate'] = (
        (totals.bots + alpha * p0)
        / (totals.visitors + alpha)
    )

    totals['has_bot'] = (totals.bots > 0).astype(float)
    totals['has_hist'] = (totals.visitors > 0).astype(float)
    by_cookie = totals.groupby('cookie_id')
    result = pd.DataFrame({
        'ih_rate_mean': by_cookie.rate.mean(),
        'ih_rate_max': by_cookie.rate.max(),
        'ih_bot_item_share': by_cookie.has_bot.mean(),
        'ih_bot_items': by_cookie.has_bot.sum(),
        'ih_hist_share': by_cookie.has_hist.mean(),
        'ih_bot_visits_log': np.log1p(by_cookie.bots.sum())
    })
    return result.reindex(who.cookie_id)

In [40]:
week_history = item_history(train, first_week, stop_day=7)
val_ids = train.loc[train.day_idx >= 7, 'cookie_id']
day0_ids = train.loc[train.day_idx == 0, 'cookie_id']

print('Таблица истории:', week_history.shape)
print('Cookie на валидации:', len(val_ids))

val_hist = week_history.loc[val_ids]
print('Без истории объявлений:', val_hist.isna().all(axis=1).sum())
print('Средняя доля объявлений с историей:',
      round(val_hist.ih_hist_share.mean(), 3))

# у первого дня нет предыдущих дней с разметкой
assert (week_history.loc[day0_ids].ih_hist_share.fillna(0) == 0).all()

print('Первые три cookie валидации:')
print(
    val_hist[
        ['ih_rate_mean', 'ih_bot_item_share', 'ih_hist_share']
    ].head(3).round(3)
)

Таблица истории: (11091, 6)
Cookie на валидации: 5161
Без истории объявлений: 70
Средняя доля объявлений с историей: 0.62
Первые три cookie валидации:
                     ih_rate_mean  ih_bot_item_share  ih_hist_share
cookie_id                                                          
ck_3eecc4692d2e8b21         0.103              0.111          0.778
ck_4634887d6ecac7ee         0.057              0.000          0.667
ck_e8d9f5d9c9eadf68         0.091              0.080          0.520


In [41]:
#тут использую все 14 дней train, для test история cookies ограничена днем 14

history_all = item_history(meta, train, stop_day=14)
test_history = history_all.loc[test.cookie_id]

print('Общая таблица:', history_all.shape)
print('Train:', history_all.loc[train.cookie_id].shape)
print('Test:', test_history.shape)
print('Без истории объявлений:', history_all.isna().all(axis=1).sum())
print('Средняя доля объявлений с историей в test:', round(test_history.ih_hist_share.mean(), 3))
print('Первые три test cookie:')
print(
    test_history[
        ['ih_rate_mean', 'ih_bot_item_share', 'ih_hist_share']
    ].head(3).round(3)
)

Общая таблица: (16000, 6)
Train: (11091, 6)
Test: (4909, 6)
Без истории объявлений: 238
Средняя доля объявлений с историей в test: 0.818
Первые три test cookie:
                     ih_rate_mean  ih_bot_item_share  ih_hist_share
cookie_id                                                          
ck_315fb710a0e371e7         0.132              0.209          0.721
ck_a76ee3b3e3e522fd         0.048              0.000          0.800
ck_94c9a4d382689e82         0.096              0.167          0.667


In [42]:
cols = [
    'ih_rate_mean', 'ih_rate_max', 'ih_bot_item_share',
    'ih_bot_items', 'ih_hist_share', 'ih_bot_visits_log'
]

assert list(history_all.columns) == cols
assert history_all.index.is_unique
assert history_all.shape == (16000, 6)
assert history_all.index.equals(pd.Index(meta.cookie_id))
assert test_history.ih_hist_share.dropna().between(0, 1).all()
first_day = history_all.loc[day0_ids]
assert (first_day.ih_bot_items.fillna(0) == 0).all()
assert (first_day.ih_hist_share.fillna(0) == 0).all()

print('Все проверки пройдены')
print('Поведенческих признаков:', base_features.shape[1])
print('Признаков репутации:', history_all.shape[1])
print('Всего признаков для финальной модели:',
      base_features.shape[1] + history_all.shape[1])

Все проверки пройдены
Поведенческих признаков: 90
Признаков репутации: 6
Всего признаков для финальной модели: 96


**Вывод**  

Добавила 6 признаков репутации объявлений. При расчёте учитываю только прошлые дни, а для валидации использую метки исключительно из обучающего периода. Финальная таблица содержит 96 признаков, включая 90 поведенческих.

**9. Валидация по времени**  

Test расположен позже train, поэтому использую временное разбиение. Проверяю модель на пяти последовательных двухдневных блоках и отдельно на второй неделе, а историю объявлений для каждого блока считаю только по обучающим дням.

In [43]:
#проверяю метрику данную в задании
from metric import precision_at_recall
from sklearn.metrics import average_precision_score, roc_auc_score

a = precision_at_recall(
    [1, 0, 1, 1, 1],
    [5, 4, 3, 2, 1]
)

b = precision_at_recall(
    [1, 1, 0, 0],
    [0.5, 0.5, 0.5, 0.5]
)

print('Первый пример:', round(a, 3))
print('Одинаковые score:', round(b, 3))

assert np.isclose(a, 0.8)
assert np.isclose(b, 0.5)

Первый пример: 0.8
Одинаковые score: 0.5


In [44]:
cv_blocks = [
    ('10–11 апреля', 4, 5),
    ('12–13 апреля', 6, 7),
    ('14–15 апреля', 8, 9),
    ('16–17 апреля', 10, 11),
    ('18–19 апреля', 12, 13),
    ('неделя -> неделя', 7, 13)
]

rows = []
for name, start, end in cv_blocks:
    fit = train.day_idx < start
    val = train.day_idx.between(start, end)
    rows.append({
        'период': name,
        'train': fit.sum(),
        'val': val.sum(),
        'ботов в val': train.loc[val, 'target'].sum()
    })
cv_info = pd.DataFrame(rows)

print(cv_info.to_string(index=False))

          период  train  val  ботов в val
    10–11 апреля   3305 1808          161
    12–13 апреля   5113 1660          120
    14–15 апреля   6773 1627          135
    16–17 апреля   8400 1453          126
    18–19 апреля   9853 1238           94
неделя -> неделя   5930 5161          408


In [45]:
#проверяю метрику на константном прогнозе

baseline = []
for name, start, end in cv_blocks:
    val = train.day_idx.between(start, end)
    y = train.loc[val, 'target']
    score = precision_at_recall(y, np.zeros(len(y)))
    baseline.append(score)
    print(f'{name}: {score:.4f}')

print('Среднее по пяти блокам:', round(np.mean(baseline[:5]), 4))
print('Неделя -> неделя:', round(baseline[-1], 4))

10–11 апреля: 0.0890
12–13 апреля: 0.0723
14–15 апреля: 0.0830
16–17 апреля: 0.0867
18–19 апреля: 0.0759
неделя -> неделя: 0.0791
Среднее по пяти блокам: 0.0814
Неделя -> неделя: 0.0791


In [46]:
#создаю истрию обьявлений для каждого фолда (важно не ислопозовать с валидации)
fold_hist = {}

for name, start, end in cv_blocks:
    past = train[train.day_idx < start]
    val = train[train.day_idx.between(start, end)]
    # метки текущего блока не должны попасть в историю объявлений
    assert set(past.cookie_id).isdisjoint(val.cookie_id)
    hist = item_history(train, past, stop_day=start)
    assert hist.index.equals(pd.Index(train.cookie_id))
    fold_hist[start] = hist
    print(start, hist.shape, 'дней истории:', past.day_idx.nunique())

4 (11091, 6) дней истории: 4
6 (11091, 6) дней истории: 6
8 (11091, 6) дней истории: 8
10 (11091, 6) дней истории: 10
12 (11091, 6) дней истории: 12
7 (11091, 6) дней истории: 7


In [47]:
# Для каждого блока обучаюсь только на предыдущих днях

def run_cv(make_model, cols, use_history=False, seeds=(0, 1, 2)):
    x_base = base_features.loc[train.cookie_id, cols]
    y = train.target.to_numpy()
    results = []
    roll_pred = []
    roll_true = []
    for name, start, end in cv_blocks:
        fit = (train.day_idx < start).to_numpy()
        val = train.day_idx.between(start, end).to_numpy()
        x = x_base
        if use_history:
            # История объявлений не содержит меток валидационного блока
            x = x.join(fold_hist[start])
        pred = np.zeros(val.sum())
        for seed in seeds:
            model = make_model(seed)
            model.fit(x.iloc[fit], y[fit])
            pred += model.predict_proba(x.iloc[val])[:, 1] / len(seeds)
        score = precision_at_recall(y[val], pred)
        results.append({'период': name, 'P@R70': score})
        # Неделю оцениваю отдельно, чтобы не считать её дважды
        if (start, end) != (7, 13):
            roll_pred.append(pred)
            roll_true.append(y[val])
    scores = pd.DataFrame(results)
    p = np.concatenate(roll_pred)
    actual = np.concatenate(roll_true)
    p70 = scores['P@R70']
    summary = {
        'roll_mean': p70.iloc[:5].mean(),
        'roll_std': p70.iloc[:5].std(ddof=0),
        'week': p70.iloc[-1],
        'pr_auc': average_precision_score(actual, p),
        'roc_auc': roc_auc_score(actual, p)
    }
    return scores, summary
print('Функция оценки готова')

Функция оценки готова


In [48]:
assert len(cv_blocks) == 6
assert len(fold_hist) == 6
assert base_features.shape == (16000, 90)

for start, hist in fold_hist.items():
    assert hist.shape == (11091, 6)
    assert hist.index.equals(pd.Index(train.cookie_id))
    assert hist.index.is_unique
    
print('Все проверки пройдены')
print('Временных блоков:', len(cv_blocks))
print('Вариантов истории:', len(fold_hist))
print('Поведенческих признаков:', base_features.shape[1])
print('Признаков репутации:', fold_hist[4].shape[1])

Все проверки пройдены
Временных блоков: 6
Вариантов истории: 6
Поведенческих признаков: 90
Признаков репутации: 6


**Вывод**  

Подготовила пять скользящих фолдов и отдельную недельную проверку. Константный baseline даёт P@R70 = 0.0814 на скользящих блоках и 0.0791 на недельном фолде. Историю объявлений пересчитываю отдельно для каждого фолда, используя только разрешённые обучающие дни.

**10. Обучение и сравнение моделей**

Сравниваю простой Random Forest, логистическую регрессию и LightGBM. Сначала использую только поведенческие признаки, затем добавляю репутацию объявлений. Все модели проверяю на одинаковых временных фолдах, основная метрика - Precision при Recall ≥ 70%.

In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Беру те же временные блоки и историю, которые подготовила на этапе 9
ROLL_FOLDS = [(start, end) for _, start, end in cv_blocks[:-1]]
WEEK_FOLD = tuple(cv_blocks[-1][1:])
ALL_FOLDS = ROLL_FOLDS + [WEEK_FOLD]
BEHAVIOUR_COLS = list(base_features.columns)
X_all = base_features.loc[train.cookie_id, BEHAVIOUR_COLS].copy()
y = train.target.to_numpy()
day = train.day_idx.to_numpy()
HISTORY = fold_hist

assert X_all.index.equals(pd.Index(train.cookie_id))
assert X_all.shape == (11091, 90)
assert len(ALL_FOLDS) == 6
assert len(HISTORY) == 6

def make_logreg(seed):
    # Заполняю пропуски и отдельно сохраняю информацию об их наличии
    return make_pipeline(
        SimpleImputer(strategy='median', add_indicator=True),
        StandardScaler(),
        LogisticRegression(
            C=0.3,
            max_iter=5000,
            random_state=seed
        )
    )

class QuickstartRF:
    # Простой baseline на количестве событий и уникальных объявлений
    def __init__(self, seed):
        self.model = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=seed,
            n_jobs=1
        )

    def fit(self, x, y):
        self.model.fit(x.fillna(0), y)
        return self

    def predict_proba(self, x):
        return self.model.predict_proba(x.fillna(0))

def show_result(name, res):
    print(
        f"{name:35s} "
        f"roll={res['roll_mean']:.4f} "
        f"week={res['week']:.4f} "
        f"PR-AUC={res['pr_auc']:.4f} "
        f"ROC-AUC={res['roc_auc']:.4f}"
    )

print('Данные и модели подготовлены')
print('Train:', X_all.shape)
print('Поведенческих признаков:', len(BEHAVIOUR_COLS))
print('Временных фолдов:', len(ALL_FOLDS))

Данные и модели подготовлены
Train: (11091, 90)
Поведенческих признаков: 90
Временных фолдов: 6


In [50]:
import sys
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

print('Python:', sys.version.split()[0])
print('LightGBM:', lgb.__version__)
print('Признаки:', base_features.shape)
print('Train:', X_all.shape)
print('Фолдов:', len(ALL_FOLDS))
print('Вариантов истории:', len(HISTORY))

assert base_features.shape == (16000, 90)
assert X_all.shape == (11091, 90)
assert len(ALL_FOLDS) == 6
assert len(HISTORY) == 6

print('Данные на месте')

Python: 3.13.0
LightGBM: 4.7.0
Признаки: (16000, 90)
Train: (11091, 90)
Фолдов: 6
Вариантов истории: 6
Данные на месте


In [51]:
LGB_PARAMS = {
    'n_estimators': 400,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.7,
    'reg_lambda': 1.0,
    'n_jobs': 1,
    'deterministic': True,
    'force_col_wise': True,
    'verbose': -1
}

def make_lgb(seed):
    return lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)

print('LightGBM готова')
print('Деревьев:', LGB_PARAMS['n_estimators'])
print('Потоков:', LGB_PARAMS['n_jobs'])

LightGBM готова
Деревьев: 400
Потоков: 1


In [52]:
def evaluate(make_model, cols, use_history=False, seeds=(0, 1, 2)):
    scores = []
    roll_pred = []
    roll_true = []

    for start, end in ALL_FOLDS:
        tr = day < start
        va = (day >= start) & (day <= end)

        x = X_all[cols]
        if use_history:
            # у каждого фолда своя история, без меток валидационных дней
            x = x.join(HISTORY[start])

        p = np.zeros(va.sum())
        fold_start = time.perf_counter()
        
        print(
            f'\nФолд {start}–{end}: '
            f'{tr.sum()} train, {va.sum()} val',
            flush=True
        )

        for seed in seeds:
            t0 = time.perf_counter()
            print(f'  seed {seed}: обучение...', flush=True)

            model = make_model(seed)
            model.fit(x.iloc[tr], y[tr])

            p += model.predict_proba(x.iloc[va])[:, 1] / len(seeds)

            spent = time.perf_counter() - t0
            print(f'  seed {seed}: готов за {spent:.1f} сек.', flush=True)

        score = precision_at_recall(y[va], p)
        scores.append(score)
        spent = time.perf_counter() - fold_start
        print(f'Фолд {start}–{end}: P@R70 = {score:.4f}, {spent:.1f} сек.', flush=True)

        # недельный фолд оцениваем отдельно
        if (start, end) != WEEK_FOLD:
            roll_pred.append(p)
            roll_true.append(y[va])
    pred = np.concatenate(roll_pred)
    actual = np.concatenate(roll_true)
    roll = np.array(scores[:-1])
    return {
        'roll_mean': roll.mean(),
        'roll_std': roll.std(),
        'week': scores[-1],
        'pr_auc': average_precision_score(actual, pred),
        'roc_auc': roc_auc_score(actual, pred),
        'folds': np.round(roll, 3).tolist()
    }

print('Функция валидации обновлена')

Функция валидации обновлена


In [53]:
# Сначала проверяю простые модели, чтобы было с чем сравнивать LightGBM

rf_res = evaluate(
    QuickstartRF,
    ['n_events', 'n_items'],
    use_history=False,
    seeds=(0,)
)
show_result('1. Random Forest', rf_res)

lr_res = evaluate(
    make_logreg,
    BEHAVIOUR_COLS,
    use_history=False,
    seeds=(0,)
)
show_result('2. LogReg, поведение', lr_res)


Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 0.5 сек.
Фолд 4–5: P@R70 = 0.1110, 0.5 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 0.6 сек.
Фолд 6–7: P@R70 = 0.0870, 0.6 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 0.8 сек.
Фолд 8–9: P@R70 = 0.1077, 0.8 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 0.9 сек.
Фолд 10–11: P@R70 = 0.0884, 0.9 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 1.0 сек.
Фолд 12–13: P@R70 = 0.1014, 1.0 сек.

Фолд 7–13: 5930 train, 5161 val
  seed 0: обучение...
  seed 0: готов за 0.7 сек.
Фолд 7–13: P@R70 = 0.0930, 0.7 сек.
1. Random Forest                    roll=0.0991 week=0.0930 PR-AUC=0.2192 ROC-AUC=0.6151

Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 0.0 сек.
Фолд 4–5: P@R70 = 0.7152, 0.0 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 0.1 сек.


In [54]:
start, end = ROLL_FOLDS[0]

tr = day < start
va = (day >= start) & (day <= end)

x = X_all[BEHAVIOUR_COLS]

print('Train:', tr.sum())
print('Validation:', va.sum())
print('Признаков:', x.shape[1])
print('Начинаю обучение...', flush=True)

t0 = time.perf_counter()

test_model = make_lgb(0)
test_model.fit(x.iloc[tr], y[tr])

spent = time.perf_counter() - t0

pred = test_model.predict_proba(x.iloc[va])[:, 1]
score = precision_at_recall(y[va], pred)

print(f'\nОдно обучение: {spent:.1f} сек.')
print(f'P@R70: {score:.4f}')
print('Проверка закончена')

Train: 3305
Validation: 1808
Признаков: 90
Начинаю обучение...

Одно обучение: 1.1 сек.
P@R70: 0.7902
Проверка закончена


In [55]:
t0 = time.perf_counter()

lgb_res = evaluate(
    make_lgb,
    BEHAVIOUR_COLS,
    use_history=False,
    seeds=(0, 1, 2)
)

print('\nОбщее время:', round((time.perf_counter() - t0) / 60, 1), 'мин.')

show_result('3. LightGBM, поведение', lgb_res)


Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.2 сек.
  seed 1: обучение...
  seed 1: готов за 1.2 сек.
  seed 2: обучение...
  seed 2: готов за 1.2 сек.
Фолд 4–5: P@R70 = 0.7651, 3.5 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.7 сек.
  seed 1: обучение...
  seed 1: готов за 1.7 сек.
  seed 2: обучение...
  seed 2: готов за 1.7 сек.
Фолд 6–7: P@R70 = 0.8571, 5.2 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 2.2 сек.
  seed 1: обучение...
  seed 1: готов за 2.1 сек.
  seed 2: обучение...
  seed 2: готов за 2.1 сек.
Фолд 8–9: P@R70 = 0.7724, 6.4 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 2.5 сек.
  seed 1: обучение...
  seed 1: готов за 2.5 сек.
  seed 2: обучение...
  seed 2: готов за 2.5 сек.
Фолд 10–11: P@R70 = 0.7946, 7.4 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 2.7 сек.
  seed 1: обучение...
  seed 1: готов за 2.7

In [56]:
t0 = time.perf_counter()
lr_hist_res = evaluate(
    make_logreg,
    BEHAVIOUR_COLS,
    use_history=True,
    seeds=(0,)
)

print('Общее время:', round((time.perf_counter() - t0) / 60, 1), 'мин.')
show_result('4. LogReg + история', lr_hist_res)


Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 0.0 сек.
Фолд 4–5: P@R70 = 0.8000, 0.1 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 0.1 сек.
Фолд 6–7: P@R70 = 0.8317, 0.1 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 0.1 сек.
Фолд 8–9: P@R70 = 0.8636, 0.1 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 0.1 сек.
Фолд 10–11: P@R70 = 0.7295, 0.1 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 0.1 сек.
Фолд 12–13: P@R70 = 0.7674, 0.1 сек.

Фолд 7–13: 5930 train, 5161 val
  seed 0: обучение...
  seed 0: готов за 0.1 сек.
Фолд 7–13: P@R70 = 0.8011, 0.1 сек.
Общее время: 0.0 мин.
4. LogReg + история                 roll=0.7985 week=0.8011 PR-AUC=0.7775 ROC-AUC=0.9343


In [57]:
t0 = time.perf_counter()
full_res = evaluate(
    make_lgb,
    BEHAVIOUR_COLS,
    use_history=True,
    seeds=(0, 1, 2)
)
print('\nОбщее время:', round((time.perf_counter() - t0) / 60, 1), 'мин.')
show_result('5. LightGBM + история', full_res)


Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.2 сек.
  seed 1: обучение...
  seed 1: готов за 1.2 сек.
  seed 2: обучение...
  seed 2: готов за 1.2 сек.
Фолд 4–5: P@R70 = 0.7902, 3.6 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.8 сек.
  seed 1: обучение...
  seed 1: готов за 1.8 сек.
  seed 2: обучение...
  seed 2: готов за 1.8 сек.
Фолд 6–7: P@R70 = 0.9043, 5.4 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 2.2 сек.
  seed 1: обучение...
  seed 1: готов за 2.2 сек.
  seed 2: обучение...
  seed 2: готов за 2.2 сек.
Фолд 8–9: P@R70 = 0.9074, 6.7 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 2.6 сек.
  seed 1: обучение...
  seed 1: готов за 2.6 сек.
  seed 2: обучение...
  seed 2: готов за 2.6 сек.
Фолд 10–11: P@R70 = 0.8411, 7.8 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 2.8 сек.
  seed 1: обучение...
  seed 1: готов за 2.9

In [58]:
#итоговое сравнение

models = [
    ('Random Forest, 2 признака', 'rf_res'),
    ('LogReg, поведение', 'lr_res'),
    ('LightGBM, поведение', 'lgb_res'),
    ('LogReg + история', 'lr_hist_res'),
    ('LightGBM + история', 'full_res')
]
rows = []
for name, var in models:
    # после прерывания обучения результат мог не сохраниться
    if var not in globals():
        print(f'Пока нет результата: {name}')
        continue

    res = globals()[var]

    rows.append({
        'Модель': name,
        'P@R70 roll': res['roll_mean'],
        'P@R70 week': res['week'],
        'PR-AUC': res['pr_auc'],
        'ROC-AUC': res['roc_auc']
    })
if rows:
    comparison = pd.DataFrame(rows).set_index('Модель')
    print(comparison.round(4).to_string())
else:
    print('Нет завершённых экспериментов')

print(f'\nЗавершено экспериментов: {len(rows)} из {len(models)}')

                           P@R70 roll  P@R70 week  PR-AUC  ROC-AUC
Модель                                                            
Random Forest, 2 признака      0.0991      0.0930  0.2192   0.6151
LogReg, поведение              0.6836      0.6313  0.7448   0.9278
LightGBM, поведение            0.7913      0.7565  0.7859   0.9329
LogReg + история               0.7985      0.8011  0.7775   0.9343
LightGBM + история             0.8604      0.8727  0.8147   0.9380

Завершено экспериментов: 5 из 5


**Вывод**  

Я сравнила Random Forest, логистическую регрессию и LightGBM на одинаковых временных фолдах. LightGBM только на поведенческих признаках получила P@R70 = 0.7913 на скользящих фолдах и 0.7565 на недельной проверке.

После добавления истории объявлений качество выросло до 0.8604 на скользящих фолдах и 0.8727 на недельной проверке. История дала модели дополнительную информацию, которой нет в поведении отдельной cookie. При этом результаты отдельных фолдов различаются, поэтому небольшие изменения метрики не считаю надёжным улучшением.

**11. Проверка признаков и сдвига данных**  


Тут я по очереди убираю группы признаков из LightGBM и сравниваю качество на одинаковых временных фолдах. Кроме этого, я отдельно проверяю, можно ли отличить вторую неделю train от test по поведенческим признакам.

In [59]:
#подготовка
groups = {
    'timing': list(time_features.columns),
    'pointer': list(pointer_feat.columns),
    'search': list(search_feat.columns),
    'covisit': list(covisit_feat.columns)
}

assert base_features.shape == (16000, 90)
assert X_all.index.equals(pd.Index(train.cookie_id))
assert len(HISTORY) == 6
assert len(BEHAVIOUR_COLS) == 90
for name, cols in groups.items():
    assert set(cols).issubset(BEHAVIOUR_COLS)
    print(name, len(cols))
print('Все данные на месте')

timing 29
pointer 10
search 7
covisit 5
Все данные на месте


In [60]:
#основная модель с одним seed
import time

abl = {}
t0 = time.perf_counter()

abl['full'] = evaluate(
    make_lgb,
    BEHAVIOUR_COLS,
    use_history=True,
    seeds=(0,)
)
print('Основная модель:')
print('P@R70 roll:', round(abl['full']['roll_mean'], 4))
print('P@R70 week:', round(abl['full']['week'], 4))
print('Время:', round((time.perf_counter() - t0) / 60, 1), 'мин.')


Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.2 сек.
Фолд 4–5: P@R70 = 0.7823, 1.2 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.8 сек.
Фолд 6–7: P@R70 = 0.9032, 1.8 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 2.2 сек.
Фолд 8–9: P@R70 = 0.9057, 2.2 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 2.6 сек.
Фолд 10–11: P@R70 = 0.8318, 2.6 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 2.8 сек.
Фолд 12–13: P@R70 = 0.8481, 2.8 сек.

Фолд 7–13: 5930 train, 5161 val
  seed 0: обучение...
  seed 0: готов за 2.2 сек.
Фолд 7–13: P@R70 = 0.8653, 2.2 сек.
Основная модель:
P@R70 roll: 0.8542
P@R70 week: 0.8653
Время: 0.2 мин.


In [61]:
#убираю временные признаки
cols = [
    col for col in BEHAVIOUR_COLS
    if col not in groups['timing']
]
print('Признаков без timing:', len(cols) + 6)
abl['timing'] = evaluate(
    make_lgb, cols,
    use_history=True,
    seeds=(0,)
)
print('\nБез timing:')
print('P@R70 roll:', round(abl['timing']['roll_mean'], 4))
print('P@R70 week:', round(abl['timing']['week'], 4))


Признаков без timing: 67

Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 0.9 сек.
Фолд 4–5: P@R70 = 0.7403, 0.9 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.2 сек.
Фолд 6–7: P@R70 = 0.7636, 1.2 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 1.5 сек.
Фолд 8–9: P@R70 = 0.8190, 1.5 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 1.7 сек.
Фолд 10–11: P@R70 = 0.7672, 1.7 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 1.9 сек.
Фолд 12–13: P@R70 = 0.7765, 1.9 сек.

Фолд 7–13: 5930 train, 5161 val
  seed 0: обучение...
  seed 0: готов за 1.5 сек.
Фолд 7–13: P@R70 = 0.7606, 1.6 сек.

Без timing:
P@R70 roll: 0.7733
P@R70 week: 0.7606


In [62]:
#убираю признаки курсора
cols = [
    col for col in BEHAVIOUR_COLS
    if col not in groups['pointer']
]
print('Признаков без pointer:', len(cols) + 6)

abl['pointer'] = evaluate(
    make_lgb, cols,
    use_history=True,
    seeds=(0,)
)
print('Без pointer:')
print('P@R70 roll:', round(abl['pointer']['roll_mean'], 4))
print('P@R70 week:', round(abl['pointer']['week'], 4))

Признаков без pointer: 86

Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.1 сек.
Фолд 4–5: P@R70 = 0.7063, 1.1 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.6 сек.
Фолд 6–7: P@R70 = 0.7727, 1.6 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 2.0 сек.
Фолд 8–9: P@R70 = 0.9143, 2.0 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 2.2 сек.
Фолд 10–11: P@R70 = 0.7177, 2.2 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 2.5 сек.
Фолд 12–13: P@R70 = 0.8148, 2.5 сек.

Фолд 7–13: 5930 train, 5161 val
  seed 0: обучение...
  seed 0: готов за 2.0 сек.
Фолд 7–13: P@R70 = 0.8148, 2.0 сек.
Без pointer:
P@R70 roll: 0.7852
P@R70 week: 0.8148


In [63]:
#теперь поиск и общие объявления
for name in ('search', 'covisit'):
    cols = [
        col for col in BEHAVIOUR_COLS
        if col not in groups[name]
    ]
    print(f'Без {name}, признаков: {len(cols) + 6}', flush=True)
    abl[name] = evaluate(
        make_lgb, cols,
        use_history=True,
        seeds=(0,)
    )
    print('P@R70 roll:', round(abl[name]['roll_mean'], 4))
    print('P@R70 week:', round(abl[name]['week'], 4))

Без search, признаков: 89

Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.1 сек.
Фолд 4–5: P@R70 = 0.7847, 1.1 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.7 сек.
Фолд 6–7: P@R70 = 0.8958, 1.7 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 2.1 сек.
Фолд 8–9: P@R70 = 0.8962, 2.1 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 2.5 сек.
Фолд 10–11: P@R70 = 0.8396, 2.5 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 2.7 сек.
Фолд 12–13: P@R70 = 0.8684, 2.7 сек.

Фолд 7–13: 5930 train, 5161 val
  seed 0: обучение...
  seed 0: готов за 2.1 сек.
Фолд 7–13: P@R70 = 0.8589, 2.1 сек.
P@R70 roll: 0.857
P@R70 week: 0.8589
Без covisit, признаков: 91

Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.2 сек.
Фолд 4–5: P@R70 = 0.8129, 1.2 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.7 сек.

In [64]:
#делаю сводную таблицу абляций

rows = []
for name, res in abl.items():
    rows.append({
        'Группа': name,
        'P@R70 roll': res['roll_mean'],
        'P@R70 week': res['week'],
        'Разница с полной': (
            res['roll_mean'] - abl['full']['roll_mean']
        )
    })
abl_table = pd.DataFrame(rows).set_index('Группа')
print(abl_table.round(4).to_string())

         P@R70 roll  P@R70 week  Разница с полной
Группа                                           
full         0.8542      0.8653            0.0000
timing       0.7733      0.7606           -0.0809
pointer      0.7852      0.8148           -0.0691
search       0.8570      0.8589            0.0027
covisit      0.8551      0.8639            0.0009


In [65]:
# Проверяю простой blend: основная LightGBM + LogReg.
# Ничего не отправляю на Stepik, сначала смотрю только временную валидацию.

mix_rows = []

for start, end in ALL_FOLDS:
    tr = day < start
    va = (day >= start) & (day <= end)
    x = X_all[BEHAVIOUR_COLS].join(HISTORY[start])
    # Основная модель — те же 3 seed, что использовались раньше
    p_lgb = np.zeros(va.sum())
    for seed in (0, 1, 2):
        model = make_lgb(seed)
        model.fit(x.iloc[tr], y[tr])
        p_lgb += model.predict_proba(x.iloc[va])[:, 1] / 3
    # Логистическая регрессия может ошибаться немного иначе,
    # поэтому проверяю, улучшает ли она ранжирование в смеси.
    lr = make_logreg(0)
    lr.fit(x.iloc[tr], y[tr])
    p_lr = lr.predict_proba(x.iloc[va])[:, 1]
    row = {
        'fold': f'{start}-{end}',
        'lgb': precision_at_recall(y[va], p_lgb)
    }
    for w in (0.05, 0.10, 0.15, 0.20, 0.25):
        p = (1 - w) * p_lgb + w * p_lr
        row[f'lr_{int(w * 100)}'] = precision_at_recall(y[va], p)
    mix_rows.append(row)
    print(f'Фолд {start}-{end} готов', flush=True)

mix = pd.DataFrame(mix_rows)
print('\nРезультаты по фолдам:')
print(mix.round(4).to_string(index=False))

Фолд 4-5 готов
Фолд 6-7 готов
Фолд 8-9 готов
Фолд 10-11 готов
Фолд 12-13 готов
Фолд 7-13 готов

Результаты по фолдам:
 fold    lgb   lr_5  lr_10  lr_15  lr_20  lr_25
  4-5 0.7902 0.7847 0.7902 0.8014 0.7902 0.7862
  6-7 0.9043 0.9053 0.9130 0.9158 0.9149 0.9140
  8-9 0.9074 0.9074 0.9074 0.9074 0.9083 0.9074
10-11 0.8411 0.8411 0.8411 0.8364 0.8364 0.8364
12-13 0.8590 0.8608 0.8590 0.8571 0.8571 0.8571
 7-13 0.8727 0.8735 0.8720 0.8693 0.8697 0.8667


In [66]:
# Первые пять фолдов — основная оценка,
# последний 7-13 — отдельная проверка "неделя -> неделя".
cols = ['lgb', 'lr_5', 'lr_10', 'lr_15', 'lr_20', 'lr_25']
check = pd.DataFrame({
    'roll': mix[cols].iloc[:5].mean(),
    'week': mix[cols].iloc[-1]
})
check['gain_roll'] = check['roll'] - check.loc['lgb', 'roll']
check['gain_week'] = check['week'] - check.loc['lgb', 'week']
print(check.round(4).to_string())

         roll    week  gain_roll  gain_week
lgb    0.8604  0.8727     0.0000     0.0000
lr_5   0.8599  0.8735    -0.0005     0.0008
lr_10  0.8622  0.8720     0.0018    -0.0008
lr_15  0.8636  0.8693     0.0032    -0.0034
lr_20  0.8614  0.8697     0.0010    -0.0030
lr_25  0.8602  0.8667    -0.0002    -0.0061


In [67]:
#обучаю отдельную модель отличать cookies второй недели train от test cookies
from sklearn.metrics import roc_auc_score

adv_ids = pd.concat([
    train.loc[train.day_idx >= 7, 'cookie_id'],
    test.cookie_id
], ignore_index=True)

adv_y = np.r_[
    np.zeros((train.day_idx >= 7).sum()),
    np.ones(len(test))
]

adv_x = base_features.loc[adv_ids, BEHAVIOUR_COLS]
adv_pred = np.zeros(len(adv_ids))
print('Cookie для проверки:', len(adv_ids))
print('Признаков:', adv_x.shape[1])

for k in range(5):
    # та же схема разбиения, что в исходном эксперименте
    val = np.arange(len(adv_ids)) % 5 == k
    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        n_jobs=1,
        deterministic=True,
        verbose=-1,
        random_state=0
    )
    model.fit(adv_x.iloc[~val], adv_y[~val])
    adv_pred[val] = model.predict_proba(adv_x.iloc[val])[:, 1]
    print(f'Фолд {k + 1}/5 готов', flush=True)

adv_auc = roc_auc_score(adv_y, adv_pred)
print('Adversarial ROC-AUC:', round(adv_auc, 3))

Cookie для проверки: 10070
Признаков: 90
Фолд 1/5 готов
Фолд 2/5 готов
Фолд 3/5 готов
Фолд 4/5 готов
Фолд 5/5 готов
Adversarial ROC-AUC: 0.497


In [68]:
#проверка user-agent
ua = events_clean.groupby('cookie_id').user_agent.first()
ua_feat = pd.DataFrame(index=pd.Index(meta.cookie_id))
ua_feat['ua_headless'] = ua.str.contains(
    'Headless', na=False
).astype(int)
ua_feat['ua_script'] = ua.str.contains(
    'curl|python|wget', case=False, na=False
).astype(int)
ua_feat['ua_app'] = ua.str.startswith(
    'Avito/', na=False
).astype(int)
ua_feat['ua_linux'] = ua.str.contains(
    'X11; Linux', na=False
).astype(int)
ua_feat['ua_chrome_ver'] = pd.to_numeric(
    ua.str.extract(r'Chrome/(\d+)')[0],
    errors='coerce'
)
ua_feat['ua_firefox_ver'] = pd.to_numeric(
    ua.str.extract(r'Firefox/(\d+)')[0],
    errors='coerce'
)
print('Размер:', ua_feat.shape)
print('Cookie с Headless:', ua_feat.ua_headless.sum())

assert ua_feat.index.is_unique
assert ua_feat.shape == (16000, 6)

Размер: (16000, 6)
Cookie с Headless: 421


In [69]:
#проверяю, дают ли UA-признаки прирост относительно прежней модели.
#обучающие дни и seed одинаковые, меняется только набор признаков.

ua_rows = []

for start, end in ALL_FOLDS:
    tr = day < start
    va = (day >= start) & (day <= end)
    hist = HISTORY[start]
    x_old = X_all[BEHAVIOUR_COLS].join(hist)
    x_new = X_all[BEHAVIOUR_COLS].join(ua_feat).join(hist)

    assert x_old.shape[1] == 96
    assert x_new.shape[1] == 102

    p_old = np.zeros(va.sum())
    p_new = np.zeros(va.sum())
    for seed in (0, 1, 2):
        old = make_lgb(seed)
        old.fit(x_old.iloc[tr], y[tr])
        p_old += old.predict_proba(x_old.iloc[va])[:, 1] / 3
        new = make_lgb(seed)
        new.fit(x_new.iloc[tr], y[tr])
        p_new += new.predict_proba(x_new.iloc[va])[:, 1] / 3
    old_score = precision_at_recall(y[va], p_old)
    new_score = precision_at_recall(y[va], p_new)
    ua_rows.append({
        'fold': f'{start}-{end}',
        'old': old_score,
        'ua': new_score,
        'gain': new_score - old_score
    })
    print(
        f'{start}-{end}: old={old_score:.4f}, '
        f'UA={new_score:.4f}',
        flush=True
    )

ua_check = pd.DataFrame(ua_rows)

print('По отдельным фолдам:')
print(ua_check.round(4).to_string(index=False))
print('Среднее по пяти фолдам:')
print(ua_check[['old', 'ua', 'gain']].iloc[:5].mean().round(4))
print('Недельный фолд:')
print(ua_check[['old', 'ua', 'gain']].iloc[-1].round(4))
better = (ua_check.gain.iloc[:5] > 0).sum()
print(f'\nUA улучшила {better} из 5 скользящих фолдов')

4-5: old=0.7902, UA=0.7847
6-7: old=0.9043, UA=0.8854
8-9: old=0.9074, UA=0.9057
10-11: old=0.8411, UA=0.8519
12-13: old=0.8590, UA=0.8701
7-13: old=0.8727, UA=0.8731
По отдельным фолдам:
 fold    old     ua    gain
  4-5 0.7902 0.7847 -0.0055
  6-7 0.9043 0.8854 -0.0188
  8-9 0.9074 0.9057 -0.0017
10-11 0.8411 0.8519  0.0107
12-13 0.8590 0.8701  0.0112
 7-13 0.8727 0.8731  0.0004
Среднее по пяти фолдам:
old     0.8604
ua      0.8596
gain   -0.0008
dtype: float64
Недельный фолд:
old     0.8727
ua      0.8731
gain    0.0004
Name: 5, dtype: float64

UA улучшила 2 из 5 скользящих фолдов


In [70]:
#проверяю два варианта LightGBM на тех же фолдах и с теми же seed.
#признаки и история объявлений остаются прежними.

variants = {
    'small_tree': {
        'num_leaves': 15,
        'min_child_samples': 35
    },
    'strong_reg': {
        'min_child_samples': 80,
        'reg_lambda': 4.0
    }
}

tuning = {}

for name, changes in variants.items():
    print(f'\nПроверяю {name}', flush=True)

    def make_variant(seed):
        params = {**LGB_PARAMS, **changes}
        return lgb.LGBMClassifier(**params, random_state=seed)

    result = evaluate(
        make_variant,
        BEHAVIOUR_COLS,
        use_history=True,
        seeds=(0, 1, 2)
    )
    tuning[name] = result
    print(
        f'{name}: roll={result["roll_mean"]:.4f}, '
        f'week={result["week"]:.4f}',
        flush=True
    )

rows = [{
    'model': 'original',
    'roll': full_res['roll_mean'],
    'week': full_res['week'],
    'better_folds': '-'
}]

for name, result in tuning.items():
    better = sum(
        a > b
        for a, b in zip(result['folds'], full_res['folds'])
    )
    rows.append({
        'model': name,
        'roll': result['roll_mean'],
        'week': result['week'],
        'better_folds': better
    })

tuning_table = pd.DataFrame(rows)

print('Сравнение:')
print(tuning_table.round(4).to_string(index=False))


Проверяю small_tree

Фолд 4–5: 3305 train, 1808 val
  seed 0: обучение...
  seed 0: готов за 1.0 сек.
  seed 1: обучение...
  seed 1: готов за 1.1 сек.
  seed 2: обучение...
  seed 2: готов за 1.0 сек.
Фолд 4–5: P@R70 = 0.7687, 3.1 сек.

Фолд 6–7: 5113 train, 1660 val
  seed 0: обучение...
  seed 0: готов за 1.3 сек.
  seed 1: обучение...
  seed 1: готов за 1.4 сек.
  seed 2: обучение...
  seed 2: готов за 1.3 сек.
Фолд 6–7: P@R70 = 0.8936, 4.0 сек.

Фолд 8–9: 6773 train, 1627 val
  seed 0: обучение...
  seed 0: готов за 1.6 сек.
  seed 1: обучение...
  seed 1: готов за 1.6 сек.
  seed 2: обучение...
  seed 2: готов за 1.6 сек.
Фолд 8–9: P@R70 = 0.8889, 4.7 сек.

Фолд 10–11: 8400 train, 1453 val
  seed 0: обучение...
  seed 0: готов за 1.7 сек.
  seed 1: обучение...
  seed 1: готов за 1.7 сек.
  seed 2: обучение...
  seed 2: готов за 1.7 сек.
Фолд 10–11: P@R70 = 0.8738, 5.2 сек.

Фолд 12–13: 9853 train, 1238 val
  seed 0: обучение...
  seed 0: готов за 1.9 сек.
  seed 1: обучение...
 

In [71]:
#проверяю сглаживание истории объявлений.
#валидация остаётся прежней, меняется только alpha.

alpha_results = {}

for alpha in (1.0, 5.0):
    print(f'\nПроверяю alpha={alpha}', flush=True)
    scores = []
    for start, end in ALL_FOLDS:
        tr = day < start
        va = (day >= start) & (day <= end)
        #для каждого фолда использую только метки прошлых дней.
        past = train[train.day_idx < start]
        hist = item_history(
            train, past,
            stop_day=start,
            alpha=alpha
        )
        x = X_all[BEHAVIOUR_COLS].join(hist)
        p = np.zeros(va.sum())
        for seed in (0, 1, 2):
            model = make_lgb(seed)
            model.fit(x.iloc[tr], y[tr])
            p += model.predict_proba(x.iloc[va])[:, 1] / 3
        score = precision_at_recall(y[va], p)
        scores.append(score)
        print(f'  {start}-{end}: {score:.4f}', flush=True)
    alpha_results[alpha] = scores

rows = [{
    'alpha': 2.0,
    'roll': full_res['roll_mean'],
    'week': full_res['week'],
    'better_folds': '-'
}]

for alpha, scores in alpha_results.items():
    better = sum(
        s > old + 0.003
        for s, old in zip(scores[:5], full_res['folds'])
    )
    rows.append({
        'alpha': alpha,
        'roll': np.mean(scores[:5]),
        'week': scores[5],
        'better_folds': better
    })

alpha_table = pd.DataFrame(rows)

print('\nСравнение сглаживания:')
print(alpha_table.round(4).to_string(index=False))


Проверяю alpha=1.0
  4-5: 0.7838
  6-7: 0.9239
  8-9: 0.9083
  10-11: 0.8558
  12-13: 0.8625
  7-13: 0.8653

Проверяю alpha=5.0
  4-5: 0.7862
  6-7: 0.9032
  8-9: 0.9083
  10-11: 0.8349
  12-13: 0.8571
  7-13: 0.8724

Сравнение сглаживания:
 alpha   roll   week better_folds
   2.0 0.8604 0.8727            -
   1.0 0.8668 0.8653            3
   5.0 0.8579 0.8724            0


In [72]:
#проверяю ансамбль из двух вариантов истории.
#для каждого фолда история использует только предыдущие дни.

blend_rows = []

for start, end in ALL_FOLDS:
    tr = day < start
    va = (day >= start) & (day <= end)
    past = train[train.day_idx < start]
    h2 = HISTORY[start]
    h1 = item_history(train, past, stop_day=start, alpha=1.0)
    x2 = X_all[BEHAVIOUR_COLS].join(h2)
    x1 = X_all[BEHAVIOUR_COLS].join(h1)
    p2 = np.zeros(va.sum())
    p1 = np.zeros(va.sum())

    for seed in (0, 1, 2):
        m2 = make_lgb(seed)
        m2.fit(x2.iloc[tr], y[tr])
        p2 += m2.predict_proba(x2.iloc[va])[:, 1] / 3
        m1 = make_lgb(seed)
        m1.fit(x1.iloc[tr], y[tr])
        p1 += m1.predict_proba(x1.iloc[va])[:, 1] / 3

    #больший вес оставляю модели с уже проверенным alpha=2
    p25 = 0.75 * p2 + 0.25 * p1
    p50 = 0.50 * p2 + 0.50 * p1

    blend_rows.append({
        'fold': f'{start}-{end}',
        'base': precision_at_recall(y[va], p2),
        'alpha1': precision_at_recall(y[va], p1),
        'mix25': precision_at_recall(y[va], p25),
        'mix50': precision_at_recall(y[va], p50)
    })
    print(f'Фолд {start}-{end} готов', flush=True)

blend_df = pd.DataFrame(blend_rows)
names = ['base', 'alpha1', 'mix25', 'mix50']
blend_check = pd.DataFrame({
    'roll': blend_df[names].iloc[:5].mean(),
    'week': blend_df[names].iloc[-1]
})
blend_check['gain_roll'] = (
    blend_check.roll - blend_check.loc['base', 'roll']
)
blend_check['gain_week'] = (
    blend_check.week - blend_check.loc['base', 'week']
)

print('Сравнение:')
print(blend_check.round(4).to_string())
print('Результаты отдельных фолдов:')
print(blend_df.round(4).to_string(index=False))

Фолд 4-5 готов
Фолд 6-7 готов
Фолд 8-9 готов
Фолд 10-11 готов
Фолд 12-13 готов
Фолд 7-13 готов
Сравнение:
          roll    week  gain_roll  gain_week
base    0.8604  0.8727     0.0000     0.0000
alpha1  0.8668  0.8653     0.0065    -0.0075
mix25   0.8588  0.8709    -0.0016    -0.0019
mix50   0.8575  0.8705    -0.0029    -0.0022
Результаты отдельных фолдов:
 fold   base  alpha1  mix25  mix50
  4-5 0.7902  0.7838 0.7847 0.7808
  6-7 0.9043  0.9239 0.9032 0.9053
  8-9 0.9074  0.9083 0.9074 0.9074
10-11 0.8411  0.8558 0.8396 0.8333
12-13 0.8590  0.8625 0.8590 0.8608
 7-13 0.8727  0.8653 0.8709 0.8705


In [73]:
%pip install catboost==1.2.8


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [74]:
from catboost import CatBoostClassifier

print('CatBoost доступна')

CatBoost доступна


In [75]:
#проверяю CatBoost как дополнение к основной LightGBM.
#для каждого фолда история содержит только предыдущие размеченные дни.

cat_rows = []

for start, end in ALL_FOLDS:
    tr = day < start
    va = (day >= start) & (day <= end)
    x = X_all[BEHAVIOUR_COLS].join(HISTORY[start])
    x = x.replace([np.inf, -np.inf], np.nan)
    p_lgb = np.zeros(va.sum())
    for seed in (0, 1, 2):
        model = make_lgb(seed)
        model.fit(x.iloc[tr], y[tr])
        p_lgb += model.predict_proba(x.iloc[va])[:, 1] / 3
    #catBoost не принимает NaN в таком числовом наборе,
    #поэтому выделяю пропуски отдельным значением.
    cat = CatBoostClassifier(
        iterations=450,
        depth=5,
        learning_rate=0.05,
        l2_leaf_reg=5,
        loss_function='Logloss',
        random_seed=0,
        thread_count=2,
        verbose=False,
        allow_writing_files=False
    )
    cat.fit(x.iloc[tr].fillna(-9999), y[tr])
    p_cat = cat.predict_proba(x.iloc[va].fillna(-9999))[:, 1]
    #вероятности, не ранги: пока проверяю один фиксированный вес.
    p_mix = 0.8 * p_lgb + 0.2 * p_cat
    cat_rows.append({
        'fold': f'{start}-{end}',
        'lgb': precision_at_recall(y[va], p_lgb),
        'cat': precision_at_recall(y[va], p_cat),
        'mix20': precision_at_recall(y[va], p_mix)
    })
    print(f'Фолд {start}-{end} готов', flush=True)

cat_cv = pd.DataFrame(cat_rows)

print('\nРезультаты по фолдам:')
print(cat_cv.round(4).to_string(index=False))

Фолд 4-5 готов
Фолд 6-7 готов
Фолд 8-9 готов
Фолд 10-11 готов
Фолд 12-13 готов
Фолд 7-13 готов

Результаты по фолдам:
 fold    lgb    cat  mix20
  4-5 0.7902 0.7862 0.7823
  6-7 0.9043 0.9231 0.9140
  8-9 0.9074 0.9223 0.9083
10-11 0.8411 0.8165 0.8364
12-13 0.8590 0.8272 0.8590
 7-13 0.8727 0.8462 0.8735


In [76]:
cat_check = pd.DataFrame({
    'roll': cat_cv[['lgb', 'cat', 'mix20']].iloc[:5].mean(),
    'week': cat_cv[['lgb', 'cat', 'mix20']].iloc[-1]
})
cat_check['gain_roll'] = (
    cat_check.roll - cat_check.loc['lgb', 'roll']
)
cat_check['gain_week'] = (
    cat_check.week - cat_check.loc['lgb', 'week']
)
better = (cat_cv.mix20.iloc[:5] > cat_cv.lgb.iloc[:5]).sum()

print(cat_check.round(4).to_string())
print(f'\nУлучшилось фолдов: {better} из 5')

         roll    week  gain_roll  gain_week
lgb    0.8604  0.8727     0.0000     0.0000
cat    0.8551  0.8462    -0.0053    -0.0266
mix20  0.8600  0.8735    -0.0004     0.0008

Улучшилось фолдов: 2 из 5


**Вывод**


На абляциях отдельно проверила временные признаки, курсор, поиск и общие объявления. Я поняла, что основные изменения качества связаны с исключением временных признаков и курсора. Проверка train -> test дала adversarial ROC-AUC около 0,49, заметного сдвига по выбранным признакам не обнаружено.

А User-Agent рассмотрела отдельно, но в итоговые признаки его не добавляю, так как в исходных экспериментах небольшой средний прирост был нестабильным между фолдами.

**12. Финальная модель**  


Здесь я обучаю LightGBM на всём train с 90 поведенческими и 6 историческими признаками. Для test использую историю объявлений только за дни train. Усредняю предсказания пяти моделей с разными random_state.

In [77]:
#готовлю данные для финальной проверки
# для train история расширяется по дням, для test доступны все 14 дней train
history_all = item_history(meta, train, stop_day=14)
beh_cols = list(base_features.columns)
hist_cols = list(history_all.columns)
cols = beh_cols + hist_cols

assert len(beh_cols) == 90
assert len(hist_cols) == 6
assert len(cols) == len(set(cols))
assert base_features.index.is_unique
assert history_all.index.is_unique

x_tr = base_features.loc[train.cookie_id].join(history_all[hist_cols])
x_te = base_features.loc[test.cookie_id].join(history_all[hist_cols])
x_tr = x_tr[cols].replace([np.inf, -np.inf], np.nan)
x_te = x_te[cols].replace([np.inf, -np.inf], np.nan)

# порядок строк должен совпадать с порядком target и cookie_id из test
assert x_tr.index.equals(pd.Index(train.cookie_id))
assert x_te.index.equals(pd.Index(test.cookie_id))
assert list(x_tr.columns) == list(x_te.columns)
assert all(pd.api.types.is_numeric_dtype(t) for t in x_tr.dtypes)

print('Train:', x_tr.shape)
print('Test:', x_te.shape)
print('Признаков:', len(cols))
print('История объявлений в test:',
      round(x_te.ih_hist_share.mean(), 3))

Train: (11091, 96)
Test: (4909, 96)
Признаков: 96
История объявлений в test: 0.818


In [78]:
#обучаю пять моделей
from time import perf_counter

pred = np.zeros(len(test))
gain = pd.Series(0.0, index=cols)
seeds = range(5)
t0 = perf_counter()

for seed in seeds:
    print(f'Модель {seed + 1}/5: обучение...', flush=True)
    start = perf_counter()
    model = make_lgb(seed)
    model.fit(x_tr, train.target.to_numpy())
    pred += model.predict_proba(x_te)[:, 1] / 5
    gain += pd.Series(
        model.booster_.feature_importance('gain'),
        index=cols
    )
    print(
        f'Модель {seed + 1}/5 готова за '
        f'{perf_counter() - start:.1f} сек.',
        flush=True
    )

print('\nОбщее время:', round(perf_counter() - t0, 1), 'сек.')
print('Предсказаний:', len(pred))
print('Минимальный score:', round(pred.min(), 6))
print('Максимальный score:', round(pred.max(), 6))

Модель 1/5: обучение...
Модель 1/5 готова за 3.2 сек.
Модель 2/5: обучение...
Модель 2/5 готова за 3.2 сек.
Модель 3/5: обучение...
Модель 3/5 готова за 3.2 сек.
Модель 4/5: обучение...
Модель 4/5 готова за 3.3 сек.
Модель 5/5: обучение...
Модель 5/5 готова за 3.3 сек.

Общее время: 16.2 сек.
Предсказаний: 4909
Минимальный score: 4.9e-05
Максимальный score: 0.998699


In [79]:
#смотрю важность признаков
importance = (gain / gain.sum()).sort_values(ascending=False)
print('Топ-15 признаков:')
print(importance.head(15).round(3).to_string())

Топ-15 признаков:
ih_rate_mean          0.105
ih_bot_item_share     0.095
ptr_x_std             0.058
ih_bot_items          0.050
dt_q90                0.045
sess_min_median_dt    0.034
logdt_std             0.027
events_per_min        0.024
sess_min_cv           0.018
dt_q75                0.018
page_mean             0.018
n_locs                0.017
n_items               0.017
item_repeat           0.017
ptr_x_max             0.017


In [80]:
#создаю и проверяю sumission.csv
import hashlib

sub = pd.DataFrame({
    'cookie_id': test.cookie_id.to_numpy(),
    'score': pred
})
sample = pd.read_csv('sample_submission.csv')

# проверя формат из задания до сохранения файла
assert list(sub.columns) == ['cookie_id', 'score']
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(test)
assert sub.cookie_id.is_unique
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.notna().all()
assert np.isfinite(sub.score).all()
assert sub.score.between(0, 1).all()

sub.to_csv('submission.csv', index=False)
with open('submission.csv', 'rb') as file:
    md5 = hashlib.md5(file.read()).hexdigest()

print('Файл submission.csv создан')
print('Размер:', sub.shape)
print('Уникальных cookie:', sub.cookie_id.nunique())
print('Уникальных score:', sub.score.nunique())
print('MD5:', md5)
print('Статистика score:')
print(sub.score.describe().round(6))

Файл submission.csv создан
Размер: (4909, 2)
Уникальных cookie: 4909
Уникальных score: 4909
MD5: 5f141a99f4f616388e1853fa8fec010e
Статистика score:
count    4909.000000
mean        0.081985
std         0.238624
min         0.000049
25%         0.001364
50%         0.003761
75%         0.013745
max         0.998699
Name: score, dtype: float64


In [ ]:
from pathlib import Path
from shutil import copyfile
from catboost import CatBoostClassifier
import hashlib

#сохраняю файл, который уже получил 0.89315 на Stepik
backup = Path('submission_0893.csv')
if not backup.exists():
    copyfile('submission.csv', backup)

#catBoost обучается на тех же 96 признаках, что и основная модель
cat = CatBoostClassifier(
    iterations=450,
    depth=5,
    learning_rate=0.05,
    l2_leaf_reg=5,
    loss_function='Logloss',
    random_seed=0,
    thread_count=2,
    verbose=False,
    allow_writing_files=False
)

cat.fit(x_tr[cols].fillna(-9999), train.target.to_numpy())
cat_pred = cat.predict_proba(x_te[cols].fillna(-9999))[:, 1]

#pred - сохранённый прогноз пяти LightGBM из этапа 12
trial_pred = 0.8 * pred + 0.2 * cat_pred

trial = pd.DataFrame({
    'cookie_id': test.cookie_id.to_numpy(),
    'score': trial_pred
})

assert len(trial) == 4909
assert trial.cookie_id.is_unique
assert trial.cookie_id.equals(test.cookie_id)
assert np.isfinite(trial.score).all()
assert trial.score.between(0, 1).all()

trial.to_csv('submission_cat20.csv', index=False)

print('Новый файл: submission_cat20.csv')
print('Размер:', trial.shape)
print('Диапазон:', round(trial.score.min(), 6),
      round(trial.score.max(), 6))

with open('submission_cat20.csv', 'rb') as f:
    print('MD5:', hashlib.md5(f.read()).hexdigest())

print('Старый файл сохранён:', backup.exists())

In [ ]:
#проверяю воспроизводимость
pred_check = np.zeros(len(test))

for seed in range(5):
    print(f'Повторный запуск {seed + 1}/5', flush=True)
    model = make_lgb(seed)
    model.fit(x_tr, train.target.to_numpy())
    pred_check += model.predict_proba(x_te)[:, 1] / 5

same = np.array_equal(pred, pred_check)
print('Предсказания совпадают:', same)
assert same, 'Повторный запуск дал другие предсказания'

Повторный запуск 1/5


**Вывод**

Я построила модель для определения автоматизированного трафика по суточной истории событий cookie, и после очистки данных получила 90 поведенческих признаков и 6 признаков репутации объявлений.

На временной валидации LightGBM с одним seed получила P@R70 = 0.8542 на пяти скользящих фолдах и 0.8653 на недельном фолде, далее при удалении временных признаков качество снизилось до 0.7733, при удалении признаков курсора до 0.7852. Удаление поиска и общих объявлений существенного улучшения не дало, поэтому сохранила полный набор признаков.

Adversarial validation дала ROC-AUC = 0.497, то естьзаметного сдвига между второй неделей train и test по выбранным признакам не обнаружено.

Для итогового submission обучила LightGBM на всём train и усреднила предсказания пяти моделей с разными random_state. Признаки репутации рассчитаны только по разрешённой истории. Файл submission.csv содержит 4 909 cookie и оценки от 0 до 1.

Основное ограничением является репутация объявлений может быть менее полезна для новых сервисов, просматривающих другой набор объявлений. Кроме того, качество на временных фолдах различается, а скрытые тестовые метки недоступны.